Copias impresas y electrónicas de *Modelado y simulación en Python* están disponibles en [No Starch Press](https://nostarch.com/modeling-and-simulation-python) y [Bookshop.org](https://bookshop.org/p/books/modeling-and-simulation-in-python-allen-b-downey/17836697?ean=9781718502161) y [Amazon](https://amzn.to/3y9UxNb).

# Glucosa e Insulina

*Modelado y Simulación en Python*

Copyright 2021 Allen Downey

Licencia: [Creative Commons Atribución-No Comercial-CompartirIgual 4.0 Internacional](https://creativecommons.org/licenses/by-nc-sa/4.0/)

In [1]:
# download modsim.py if necessary

from os.path import basename, exists

def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve
        local, _ = urlretrieve(url, filename)
        print('Downloaded ' + local)
    
download('https://github.com/AllenDowney/ModSimPy/raw/master/modsim.py')

In [3]:
# import functions from modsim

from modsim import *

Este capítulo está disponible como un cuaderno Jupyter donde puede leer el texto, ejecutar el código y trabajar en los ejercicios. 
Haga clic aquí para acceder a los cuadernos: <https://allendowney.github.io/ModSimPy/>.

El capítulo anterior presenta el modelo mínimo del sistema glucosa-insulina e introduce una herramienta que necesitamos para implementarlo: la interpolación.

En este capítulo, implementaremos el modelo de dos maneras:

* Comenzaremos reescribiendo las ecuaciones diferenciales como ecuaciones en diferencias; luego resolveremos las ecuaciones en diferencias usando una versión de `run_simulation` similar a la que hemos usado en capítulos anteriores.

* Luego usaremos una nueva función SciPy, llamada `solve_ivp`, para resolver la ecuación diferencial usando un mejor algoritmo.

Veremos que `solve_ivp` es más rápido y preciso que `run_simulation`.
Como resultado, lo usaremos para los modelos del resto del libro.

La siguiente celda descarga los datos.

In [4]:
download('https://github.com/AllenDowney/ModSim/raw/main/data/' +
         'glucose_insulin.csv')

Podemos usar Pandas para leer los datos.

In [5]:
from pandas import read_csv

data = read_csv('glucose_insulin.csv', index_col='time');

## Implementando el modelo

Para empezar, supongamos que se conocen los parámetros del modelo.
Implementaremos el modelo y lo usaremos para generar series de tiempo para `G` y `X`. 
Luego veremos cómo podemos elegir parámetros que hagan que la simulación se ajuste a los datos.

Aquí están los parámetros.

In [6]:
G0 = 270
k1 = 0.02
k2 = 0.02
k3 = 1.5e-05

Pondré estos valores en una secuencia que pasaremos a `make_system`:

In [7]:
params = G0, k1, k2, k3

Aquí hay una versión de `make_system` que toma `params` y `data` como parámetros.

In [8]:
def make_system(params, data):
    G0, k1, k2, k3 = params
    
    t_0 = data.index[0]
    t_end = data.index[-1]
    
    Gb = data.glucose[t_0]
    Ib = data.insulin[t_0]
    I = interpolate(data.insulin)
    
    init = State(G=G0, X=0)
    
    return System(init=init, params=params,
                  Gb=Gb, Ib=Ib, I=I,
                  t_0=t_0, t_end=t_end, dt=2)

`make_system` obtiene `t_0` y `t_end` de los datos. 
Utiliza las mediciones en `t=0` como niveles basales, `Gb` y `Ib`. 
Y utiliza el parámetro `G0` como valor inicial para `G`. entonces 
empaqueta todo en un objeto `System`.

In [9]:
system = make_system(params, data)

## La función de actualización

El modelo mínimo se expresa en términos de ecuaciones diferenciales:

$$\frac{dG}{dt} = -k_1 \left[ G(t) - G_b \right] - X(t) G(t)$$

$$\frac{dX}{dt} = k_3 \left[I(t) - I_b \right] - k_2 X(t)$$ 

Para simular este sistema, los reescribiremos como ecuaciones en diferencias. 
Si multiplicamos ambos lados por $dt$, tenemos:

$$dG = \left[ -k_1 \left[ G(t) - G_b \right] - X(t) G(t) \right] dt$$

$$dX = \left[ k_3 \left[I(t) - I_b \right] - k_2 X(t) \right] dt$$ 

Si pensamos en $dt$ como un pequeño paso en el tiempo, estas ecuaciones nos dicen cómo calcular los cambios correspondientes en $G$ y $X$.
Aquí hay una función de actualización que calcula estos cambios:

In [10]:
def update_func(t, state, system):
    G, X = state
    G0, k1, k2, k3 = system.params 
    I, Ib, Gb = system.I, system.Ib, system.Gb
    dt = system.dt
        
    dGdt = -k1 * (G - Gb) - X*G
    dXdt = k3 * (I(t) - Ib) - k2 * X
    
    G += dGdt * dt
    X += dXdt * dt

    return State(G=G, X=X)

Como es habitual, la función de actualización toma una marca de tiempo, un objeto `State` y un objeto `System` como parámetros. La primera línea utiliza asignación múltiple para extraer los valores actuales de `G` y `X`.

Las siguientes líneas descomprimen los parámetros que necesitamos del `System`
objeto.

Para calcular las derivadas `dGdt` y `dXdt` traducimos las ecuaciones de la notación matemática a Python.
Luego, para realizar la actualización, multiplicamos cada derivada por el paso de tiempo `dt`, que en este ejemplo es de 2 min. 

El valor de retorno es un objeto `State` con los nuevos valores de `G` y `X`.

Antes de ejecutar la simulación, es una buena idea ejecutar la actualización.
funcionar con las condiciones iniciales:

In [11]:
update_func(system.t_0, system.init, system)

Si se ejecuta sin errores y no hay nada obviamente malo en los resultados, estamos listos para ejecutar la simulación. 

## Ejecutando la simulación

Usaremos la siguiente versión de `run_simulation`:

In [12]:
def run_simulation(system, update_func):    
    t_array = linrange(system.t_0, system.t_end, system.dt)
    n = len(t_array)
    
    frame = TimeFrame(index=t_array, 
                      columns=system.init.index)
    frame.iloc[0] = system.init
    
    for i in range(n-1):
        t = t_array[i]
        state = frame.iloc[i]
        frame.iloc[i+1] = update_func(t, state, system)
    
    return frame

Esta versión es similar a la que usamos para el problema de enfriamiento del café.
La mayor diferencia es que crea y devuelve un `TimeFrame`, que contiene una columna para cada variable de estado, en lugar de un `TimeSeries`, que solo puede almacenar una variable de estado.

Cuando creamos `TimeFrame`, usamos `index` para indicar que el índice es la matriz de marcas de tiempo, `t_array` y `columns` para indicar que los nombres de las columnas son las variables de estado que obtenemos de `init`.

Podemos ejecutarlo así: 

In [13]:
results = run_simulation(system, update_func)

El resultado es un `TimeFrame` con una fila para cada paso de tiempo y una columna para cada una de las variables de estado, `G` y `X`.
Estos son los primeros pasos.

In [14]:
results.head()

El siguiente gráfico muestra los niveles de glucosa simulados del modelo junto con los datos medidos.

In [15]:
data.glucose.plot(style='o', alpha=0.5, label='glucose data')
results.G.plot(style='-', color='C0', label='simulation')

decorate(xlabel='Time (min)',
         ylabel='Concentration (mg/dL)')

Con los parámetros que elegí, el modelo se ajusta bien a los datos excepto durante los primeros minutos después de la inyección.
Pero no esperamos que al modelo le vaya bien en esta parte de la serie temporal.

El problema es que el modelo es *no espacial*; es decir, no
tener en cuenta diferentes concentraciones en diferentes partes del
cuerpo. En cambio, supone que las concentraciones de glucosa e insulina en la sangre, y de insulina en el líquido tisular, son las mismas en todo el cuerpo. Esta forma de representar el cuerpo es conocida entre los expertos como modelo de la “bolsa de sangre”.

Inmediatamente después de la inyección, la glucosa inyectada tarda un tiempo en
circular. Durante ese tiempo, no esperamos que se desarrolle un modelo no espacial.
exacto. Por esta razón, no debemos tomarnos demasiado en serio el valor estimado de `G0`; es útil para ajustar el modelo, pero no pretende corresponder a una cantidad física mensurable.

El siguiente gráfico muestra niveles de insulina simulados en el hipotético "compartimento remoto", que está en unidades no especificadas.

In [16]:
results.X.plot(color='C1', label='remote insulin')

decorate(xlabel='Time (min)', 
         ylabel='Concentration (arbitrary units)')

Recuerde que `X` representa la concentración de insulina en el "compartimento remoto", que se cree que es líquido tisular, por lo que no podemos compararlo con la concentración medida de insulina en la sangre.

`X` aumenta rápidamente después de la inyección inicial y luego disminuye a medida que disminuye la concentración de glucosa.  Cualitativamente, este comportamiento es el esperado, pero como `X` no es una cantidad observable, no podemos validar cuantitativamente esta parte del modelo.

## Resolver ecuaciones diferenciales

Para implementar el modelo mínimo, reescribimos las ecuaciones diferenciales como ecuaciones en diferencias con un paso de tiempo finito, `dt`.
Cuando $dt$ es muy pequeño, o más precisamente *infinitesimal*, las ecuaciones en diferencias son las mismas que las ecuaciones diferenciales.
Pero en nuestras simulaciones, $dt$ es 2 min, lo cual no es muy pequeño y definitivamente no es infinitesimal. 

De hecho, las simulaciones suponen que las derivadas $dG/dt$ y $dX/dt$ son constantes durante cada paso de tiempo de 2 minutos.
Este método, que evalúa derivadas en pasos de tiempo discretos y supone que son constantes entre ellos, se denomina *método de Euler* (consulte <http://modsimpy.com/euler>).

El método de Euler es suficientemente bueno para muchos problemas, pero a veces no es muy preciso.
En ese caso, normalmente podemos hacerlo más preciso disminuyendo el tamaño de `dt`.
Pero entonces no es muy eficiente.

Existen otros métodos que son más precisos y eficientes que el método de Euler.
SciPy proporciona varios de ellos envueltos en una función llamada `solve_ivp`.
`ivp` significa *problema de valor inicial*, que es el término para problemas como los que hemos estado resolviendo, donde se nos dan las condiciones iniciales y tratamos de predecir qué sucederá.

La biblioteca ModSim proporciona una función llamada `run_solve_ivp` que hace que `solve_ivp` sea un poco más fácil de usar.

Para usarlo, tenemos que proporcionar una *función de pendiente*, que es similar a una función de actualización; de hecho, toma los mismos parámetros: una marca de tiempo, un objeto `State` y un objeto `System`.

Aquí hay una función de pendiente que evalúa las ecuaciones diferenciales del modelo mínimo.

In [17]:
def slope_func(t, state, system):
    G, X = state
    G0, k1, k2, k3 = system.params 
    I, Ib, Gb = system.I, system.Ib, system.Gb
        
    dGdt = -k1 * (G - Gb) - X*G
    dXdt = k3 * (I(t) - Ib) - k2 * X
    
    return dGdt, dXdt

`slope_func` es un poco más simple que `update_func` porque calcula solo las derivadas, es decir, las pendientes. No realiza las actualizaciones; el solucionador los hace por nosotros.

Ahora podemos llamar a `run_solve_ivp` así:

In [18]:
results2, details = run_solve_ivp(system, slope_func,
                                  t_eval=results.index)

`run_solve_ivp` es similar a `run_simulation`: se necesita un `System`
objeto y una función de pendiente como parámetros.

El tercer argumento, `t_eval`, es opcional; especifica dónde se debe evaluar la solución.

Devuelve dos valores: un `TimeFrame`, que asignamos a `results2`, y un objeto `OdeResult`, que asignamos a `details`.

El objeto `OdeResult` contiene información sobre cómo se ejecutó el solucionador, incluido un código de éxito y un mensaje de diagnóstico.

In [19]:
details.success

In [20]:
details.message

Es importante comprobar estos mensajes después de ejecutar el solucionador, en caso de que algo salga mal.

El `TimeFrame` tiene una fila para cada paso de tiempo y una columna para cada variable de estado. En este ejemplo, las filas son tiempos de 0 a 182 minutos; las columnas son las variables de estado, `G` y `X`.
Estos son los primeros pasos de tiempo:

In [21]:
results2.head()

Debido a que usamos `t_eval=results.index`, las marcas de tiempo en `results2` son las mismas que en `results`, lo que las hace más fáciles de comparar.

La siguiente figura muestra los resultados de `run_solve_ivp` junto con los resultados de `run_simulation`:

In [22]:
results.G.plot(style='--', label='simulation')
results2.G.plot(style='-', label='solve ivp')

decorate(xlabel='Time (min)',
         ylabel='Concentration (mg/dL)')

Las diferencias apenas son visibles.
Podemos calcular las diferencias relativas de esta manera:



In [23]:
diff = results.G - results2.G
percent_diff = diff / results2.G * 100

Y podemos usar `describe` para calcular estadísticas resumidas:

In [24]:
percent_diff.abs().describe()

La diferencia relativa media es de alrededor del 0,65% y el máximo es de poco más del 1%.
Aquí están los resultados para `X`.

In [25]:
results.X.plot(style='--', label='simulation')
results2.X.plot(style='-', label='solve ivp')

decorate(xlabel='Time (min)', 
         ylabel='Concentration (arbitrary units)')

Estas diferencias son un poco mayores, especialmente al principio.

Si utilizamos `run_simulation` con pasos de tiempo más pequeños, los resultados son más precisos, pero tardan más en calcularse.
Para algunos problemas, podemos encontrar un valor de `dt` que produzca resultados precisos en un tiempo razonable. Sin embargo, si `dt` es *demasiado* pequeño, los resultados pueden volver a ser inexactos. Por eso puede resultar complicado hacerlo bien.

La ventaja de `run_solve_ivp` es que elige el tamaño del paso automáticamente para equilibrar precisión y eficiencia.
Puede utilizar argumentos de palabras clave para ajustar este equilibrio, pero la mayoría de las veces los resultados son lo suficientemente precisos y el cálculo es lo suficientemente rápido, sin ninguna intervención.

In [26]:
diff = results.G - results2.X
percent_diff = diff / results2.X * 100
percent_diff.abs().describe()

## Resumen

En este capítulo, implementamos el modelo mínimo de glucosa de dos maneras, usando `run_simulation` y `run_solve_ivp`, y comparamos los resultados.
Descubrimos que en este ejemplo, `run_simulation`, que utiliza el método de Euler, probablemente sea lo suficientemente bueno.
Pero pronto veremos ejemplos en los que no es así.

Hasta ahora hemos asumido que se conocen los parámetros del sistema, pero en la práctica eso no es cierto.
Como uno de los estudios de caso del próximo capítulo, tendrá la oportunidad de ver de dónde provienen esos parámetros.

## Ejercicios

Este capítulo está disponible como un cuaderno Jupyter donde puede leer el texto, ejecutar el código y trabajar en los ejercicios. 
Puedes acceder a los cuadernos en <https://allendowney.github.io/ModSimPy/>.

### Ejercicio 1

Nuestra solución a las ecuaciones diferenciales es sólo aproximada porque utilizamos un tamaño de paso finito, `dt=2` minutos.
Si reducimos el tamaño del paso, esperamos que la solución sea más precisa.  Ejecute la simulación con `dt=1` y compare los resultados.  ¿Cuál es el mayor error relativo entre las dos soluciones?

In [27]:
# Solution goes here

In [28]:
# Solution goes here

In [29]:
# Solution goes here